Magnitude Pruning Step-1: Calibrating, Pruning and Saving Model Weight Shards

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "codellama/CodeLlama-7b-Instruct-hf"
sparsity = 0.40                       # fraction of weights zeroed in each linear layer
save_dir = "/content/pruned_base"     # pruned, not yet fine-tuned

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set the Colab runtime to T4.")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="cuda")
model.eval()

# --- Magnitude pruning: per weight matrix, zero the `sparsity` fraction with the smallest |w|.
# Scope: all nn.Linear layers inside the decoder blocks (attention q/k/v/o and MLP gate/up/down).
# Embeddings and lm_head are left dense, as is standard.
pruned_total, zero_total = 0, 0
with torch.no_grad():
    for layer in model.model.layers:
        for name, module in layer.named_modules():
            if isinstance(module, torch.nn.Linear):
                W = module.weight.data
                k = int(W.numel() * sparsity)
                threshold = W.abs().float().view(-1).kthvalue(k).values
                W[W.abs() <= threshold.to(W.dtype)] = 0
                pruned_total += W.numel()
                zero_total += (W == 0).sum().item()
    torch.cuda.empty_cache()

print(f"Target sparsity: {sparsity:.0%}")
print(f"Measured sparsity in pruned layers: {zero_total / pruned_total:.2%} "
      f"({zero_total:,} of {pruned_total:,} weights)")
all_params = sum(p.numel() for p in model.parameters())
print(f"Share of all model parameters zeroed: {zero_total / all_params:.2%}")

# --- Coherence check: one greedy generation of the task prompt (untracked)
Code = '''from typing import List
def below_zero(operations: List[int]) -> bool:
    """ You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False'''

prompt = ("Generate Google-style docstring for the following Python function: {code}. "
          "Only output the docstring and function code.").format(code=Code)   # P-2

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
n_new = out.shape[1] - inputs["input_ids"].shape[1]
print(f"\n--- Coherence check ({n_new} tokens, limit 300) ---")
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

# --- Save the pruned model (dense format: zeros are still stored).
# Small shards: each shard is staged in system RAM while it is written, and one 13.5 GB
# shard would exceed Colab's RAM. The tokenizer is not saved; later steps load it from
# the original model, since pruning never changes it.
model.save_pretrained(save_dir, max_shard_size="2GB")
print(f"\nPruned model saved to {save_dir}")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Target sparsity: 40%
Measured sparsity in pruned layers: 40.08% (2,595,783,122 of 6,476,005,376 weights)
Share of all model parameters zeroed: 38.52%

--- Coherence check (227 tokens, limit 300) ---
 Sure, here is the docstring and function code for the function you specified:

```
def below_zero(operations: List[int]) -> bool:
    """
    You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False
```
The docstring explains that the function takes a list of integers as input, and returns a boolean value indicating whether the balance of the account falls below zero at any point during the operations. The function iterates through the list of operations using a for loop

Writing model shards:   0%|          | 0/7 [00:00<?, ?it/s]


Pruned model saved to /content/pruned_base


Checking on the saved shards

In [2]:
!ls /content/pruned_base
!du -sh /content/pruned_base

config.json			  model-00004-of-00007.safetensors
generation_config.json		  model-00005-of-00007.safetensors
model-00001-of-00007.safetensors  model-00006-of-00007.safetensors
model-00002-of-00007.safetensors  model-00007-of-00007.safetensors
model-00003-of-00007.safetensors  model.safetensors.index.json
13G	/content/pruned_base
